In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

In [2]:
import pandas as pd

# 1. 파일 경로 설정
file_path = "C:/data/서울특별시 공공자전거 대여이력 정보_2026/서울특별시 공공자전거 대여이력 정보_2601.csv"

# 2. 데이터 불러오기 
df = pd.read_csv(file_path, encoding='cp949')

print(f"원래 데이터 크기 (행, 열): {df.shape}")

# 3. 결측치가 포함된 행 모두 제거
df_cleaned = df.dropna()

print(f"결측치 제거 후 데이터 크기 (행, 열): {df_cleaned.shape}")

원래 데이터 크기 (행, 열): (1640582, 16)
결측치 제거 후 데이터 크기 (행, 열): (1161271, 16)


In [3]:
df.info

<bound method DataFrame.info of              자전거번호                 대여일시  대여 대여소번호           대여 대여소명  대여거치대  \
0        SPB-65030  2026-01-01 00:00:13      1729     도봉한신아파트 버스정류장      0   
1        SPB-46777  2026-01-01 00:00:40      4136  롯데캐슬SKY-L65 D동 앞      0   
2        SPB-67662  2026-01-01 00:01:21      2139          서울신성초등학교      0   
3        SPB-69593  2026-01-01 00:01:50      6188             금호아파트      0   
4        SPB-55851  2026-01-01 00:03:32      4892        성내유수지 체육공원      0   
...            ...                  ...       ...               ...    ...   
1640577  SPB-73200  2026-01-31 00:52:06      3534      건대입구역 5번출구 뒤      0   
1640578  SPB-40432  2026-01-31 00:15:29      6058       서울도시건축전시관 옆      0   
1640579  SPB-31031  2026-01-31 21:42:23      4527      목동신시가지아파트6단지      0   
1640580  SPB-67592  2026-01-31 19:12:41       654          전농동 텃골공원      0   
1640581  SPB-40354  2026-01-31 18:07:22      3507             어린이회관      0   

                        반납일시  반

In [4]:
print(df.columns)

Index(['자전거번호', '대여일시', '대여 대여소번호', '대여 대여소명', '대여거치대', '반납일시', '반납대여소번호',
       '반납대여소명', '반납거치대', '이용시간(분)', '이용거리(M)', '생년', '성별', '이용자종류', '대여대여소ID',
       '반납대여소ID'],
      dtype='object')


In [3]:
# 1. 작업용 데이터프레임 복사 (원본 df는 보존)
df_cleaned = df.copy()

# 2. 결측치(NaN 또는 \N 등)가 포함된 행 모두 제거
import numpy as np
df_cleaned.replace(r'\\N', np.nan, regex=True, inplace=True)
df_cleaned = df_cleaned.dropna()

# 3. 삭제할 컬럼 리스트 지정 (위치 번호/명 및 거치대 관련 열 포함)
cols_to_drop = [
    '대여 대여소번호', 
    '대여 대여소명', 
    '반납대여소번호', 
    '반납대여소명', 
    '대여거치대', 
    '반납거치대',
    '생년', 
    '성별', 
    '이용자종류', 
    '자전거구분'
]

# 4. 지정한 열들 삭제
df_cleaned.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 5. 결과 확인
print(f"최종 전처리된 데이터 크기 (행, 열): {df_cleaned.shape}")
print("남아있는 컬럼 목록:")
print(list(df_cleaned.columns))

# 6. 최종 전처리된 파일만 별도의 이름으로 저장
output_path = "C:/data/서울특별시 공공자전거 대여이력 정보_2026/cleaned_2601_final.csv"
df_cleaned.to_csv(output_path, index=False, encoding='cp949')

print(f"\n[완료] 전처리된 파일이 아래 경로에 저장되었습니다:\n{output_path}")

NameError: name 'df' is not defined

In [6]:
df_cleaned.head()

,자전거번호,대여일시,반납일시,이용시간(분),이용거리(M),대여대여소ID,반납대여소ID
1,SPB-46777,2026-01-01 00:00:40,2026-01-01 00:02:30,1,430.00,ST-3286,ST-392
2,SPB-67662,2026-01-01 00:01:21,2026-01-01 00:02:59,1,296.86,ST-951,ST-3143
4,SPB-55851,2026-01-01 00:03:32,2026-01-01 00:04:36,1,0.00,ST-3005,ST-3005
6,SPB-33110,2026-01-01 00:00:10,2026-01-01 00:04:54,4,461.41,ST-3006,ST-2906
7,SPB-46542,2026-01-01 00:02:04,2026-01-01 00:05:20,3,506.25,ST-1990,ST-291


In [7]:
import pandas as pd
import numpy as np

# 1. 처리할 월 리스트 정의 (1월부터 6월까지 전체)
months = ['01', '02', '03', '04', '05', '06']

# 2. 삭제할 불필요한 컬럼 리스트
cols_to_drop = [
    '대여 대여소번호', 
    '대여 대여소명', 
    '반납대여소번호', 
    '반납대여소명', 
    '대여거치대', 
    '반납거치대',
    '생년', 
    '성별', 
    '이용자종류', 
    '자전거구분'
]

# 3. 폴더 경로 설정
folder_path = "C:/data/서울특별시 공공자전거 대여이력 정보_2026/"

# 4. 기준이 될 1월 파일의 컬럼 구조 확인
base_file_path = f"{folder_path}서울특별시 공공자전거 대여이력 정보_2601.csv"
base_df = pd.read_csv(base_file_path, encoding='cp949')
base_columns = list(base_df.columns)

print(f"=== [기준] 2601.csv 컬럼 구조 확인완료 (총 {len(base_columns)}개) ===\n")

# 5. 1월부터 6월까지 반복문 실행
for m in months:
    # 실제 파일 이름 형식 맞추기 (예: 서울특별시 공공자전거 대여이력 정보_2601.csv)
    file_name = f"서울특별시 공공자전거 대여이력 정보_26{m}.csv"
    file_path = f"{folder_path}{file_name}"
    
    print(f"[{m}월 파일 처리 중] : {file_name}")
    
    try:
        # 데이터 로드 (인코딩 에러 방지 안전장치)
        try:
            df = pd.read_csv(file_path, encoding='cp949')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='utf-8')
            
        # 컬럼명 일치 여부 확인
        if list(df.columns) != base_columns:
            print(f"  ⚠️ 경고: {file_name} 파일의 컬럼 구성이 기준과 다릅니다!")
        else:
            print(f"  - 컬럼명 일치 확인됨")
            
        # 6.결측치(NaN 또는 \N)만 제거한 원본 유지용 파일 저장
        df_no_na = df.copy()
        df_no_na.replace(r'\\N', np.nan, regex=True, inplace=True)
        df_no_na.dropna(inplace=True)
        
        cleaned_raw_path = f"{folder_path}cleaned_raw_26{m}.csv"
        df_no_na.to_csv(cleaned_raw_path, index=False, encoding='cp949')
        print(f"  - 결측치 제거 원본 저장: {cleaned_raw_path}")
        
        # 7.불필요한 컬럼 삭제 후 최종 전처리 파일 저장
        df_final = df_no_na.copy()
        df_final.drop(columns=cols_to_drop, inplace=True, errors='ignore')
        
        final_output_path = f"{folder_path}cleaned_26{m}_final.csv"
        df_final.to_csv(final_output_path, index=False, encoding='cp949')
        print(f"  - 최종 전처리 파일 저장: {final_output_path}\n")
        
    except FileNotFoundError:
        print(f"  ❌ 에러: 파일을 찾을 수 없습니다. 경로를 확인해주세요: {file_path}\n")

print("=== 1월부터 6월까지 전체 파일의 전처리 및 저장이 완료되었습니다! ===")

=== [기준] 2601.csv 컬럼 구조 확인완료 (총 16개) ===

[01월 파일 처리 중] : 서울특별시 공공자전거 대여이력 정보_2601.csv
  - 컬럼명 일치 확인됨
  - 결측치 제거 원본 저장: C:/data/서울특별시 공공자전거 대여이력 정보_2026/cleaned_raw_2601.csv
  - 최종 전처리 파일 저장: C:/data/서울특별시 공공자전거 대여이력 정보_2026/cleaned_2601_final.csv

[02월 파일 처리 중] : 서울특별시 공공자전거 대여이력 정보_2602.csv
  - 컬럼명 일치 확인됨
  - 결측치 제거 원본 저장: C:/data/서울특별시 공공자전거 대여이력 정보_2026/cleaned_raw_2602.csv
  - 최종 전처리 파일 저장: C:/data/서울특별시 공공자전거 대여이력 정보_2026/cleaned_2602_final.csv

[03월 파일 처리 중] : 서울특별시 공공자전거 대여이력 정보_2603.csv
  - 컬럼명 일치 확인됨
  - 결측치 제거 원본 저장: C:/data/서울특별시 공공자전거 대여이력 정보_2026/cleaned_raw_2603.csv
  - 최종 전처리 파일 저장: C:/data/서울특별시 공공자전거 대여이력 정보_2026/cleaned_2603_final.csv

[04월 파일 처리 중] : 서울특별시 공공자전거 대여이력 정보_2604.csv
  - 컬럼명 일치 확인됨
  - 결측치 제거 원본 저장: C:/data/서울특별시 공공자전거 대여이력 정보_2026/cleaned_raw_2604.csv
  - 최종 전처리 파일 저장: C:/data/서울특별시 공공자전거 대여이력 정보_2026/cleaned_2604_final.csv

[05월 파일 처리 중] : 서울특별시 공공자전거 대여이력 정보_2605.csv
  - 컬럼명 일치 확인됨
  - 결측치 제거 원본 저장: C:/data/서울특별시 공공자전거 대여이력 정보_2026/cleaned_raw_2605.cs